# 02 · Nettoyage et construction du dataset global

Chaque traitement répond à une anomalie du diagnostic (notebook 01). Le code est dans `src/fao_pipeline.py`
pour être réutilisable et testé (`tests/test_pipeline.py`) ; ce notebook l'exécute et contrôle le résultat.

**Décisions appliquées**

1. `"<0.1"` → pas de valeur imposée, mais une borne haute du taux (`Taux_max_pct`).
2. Vides NR / NV → statut conservé (`Statut_sous_nutrition`), imputés au notebook 04.
3. Agrégat « Chine » (351) exclu.
4. Période de sous-alimentation **2012-2014**, centrée sur l'année 2013 des bilans alimentaires.
5. Variation de stock négative conservée ; disponibilité négative (avoine, Japon) exclue des sommes.
6. Jointure **à gauche** depuis le socle de 175 zones (bilan alimentaire + population) ; zones perdues listées.

In [1]:
import sys
from pathlib import Path

import pandas as pd

RACINE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RACINE / "src"))
import fao_pipeline as fp

sources = fp.charger(RACINE / "data")
df, journal = fp.construire(sources, periode="2012-2014", ref_regions=RACINE / "data" / "ref_zones_regions.csv")
print(f"{len(df)} zones x {df.shape[1]} colonnes")

174 zones x 22 colonnes


## 1. Contrôles de la jointure

On fige les hypothèses avec `assert` : si une source change, le notebook échoue bruyamment au lieu de
produire un résultat faux en silence.

In [2]:
assert df["Code zone"].is_unique, "une zone apparaît deux fois"
assert len(df) == journal["zones_socle"] - 1, "seul l'agrégat Chine doit sortir du socle"
assert df["Kcal_total"].notna().all() and df["Population_millions"].notna().all()
assert df["Taux_sous_nutrition_pct"].dropna().between(0, 100).all()
assert df["ISO3"].notna().all() and df["Region"].notna().all()
print("Tous les contrôles passent.")
print(f"Socle : {journal['zones_socle']} zones ; dataset final : {journal['zones_finales']} zones (agrégat Chine exclu)")

Tous les contrôles passent.
Socle : 175 zones ; dataset final : 174 zones (agrégat Chine exclu)


In [3]:
print(f"Zones suivies pour la sous-alimentation mais sans bilan alimentaire ({len(journal['zones_sans_bilan_alimentaire'])}) :")
print(", ".join(journal["zones_sans_bilan_alimentaire"]))

Zones suivies pour la sous-alimentation mais sans bilan alimentaire (29) :
Andorre, Bahreïn, Bhoutan, Burundi, Comores, Groenland, Guinée équatoriale, Libye, Micronésie (États fédérés de), Nauru, Nioué, Palaos, Palestine, Papouasie-Nouvelle-Guinée, Porto Rico, Qatar, République arabe syrienne, République démocratique du Congo, Samoa américaines, Seychelles, Singapour, Somalie, Soudan du Sud, Tokélaou, Tonga, Tuvalu, Érythrée, Îles Cook, Îles Marshall


## 2. Statut de la sous-nutrition par zone

In [4]:
statuts = df.groupby("Statut_sous_nutrition").agg(
    zones=("Zone", "size"),
    population_millions=("Population_millions", "sum"),
    kcal_moyenne=("Kcal_total", "mean"),
).round(0)
statuts["part_population_pct"] = (statuts["population_millions"] / statuts["population_millions"].sum() * 100).round(1)
statuts

,zones,population_millions,kcal_moyenne,part_population_pct
Statut_sous_nutrition,,,,
"< 0,1 million (borne)",23,14.00,2 853.00,0.20
mesurée,96,5 367.00,2 625.00,76.70
non disponible (NV),8,13.00,2 524.00,0.20
non rapportée (NR),47,1 604.00,3 353.00,22.90


In [5]:
# Exemple de borne : pour un petit pays, "<0.1 million" ne dit presque rien du taux
df.loc[df["Sous_alim_borne"], ["Zone", "Population_millions", "Taux_max_pct"]].sort_values("Taux_max_pct").head(8)

,Zone,Population_millions,Taux_max_pct
96,Macédoine du Nord,2.11,4.75
162,Trinité-et-Tobago,1.34,7.46
52,Estonie,1.29,7.77
104,Maurice,1.24,8.04
37,Chypre,1.14,8.76
57,Fidji,0.88,11.35
69,Guyana,0.80,12.50
34,Chine - RAS de Macao,0.57,17.67


**Lecture.** Pour Chypre ou l'Estonie, la borne situe le taux sous 10 % environ. Pour la Dominique ou Kiribati,
elle autorise presque n'importe quelle valeur. Remplacer `"<0.1"` par 0,05 million aurait donné 69 % de
sous-nutrition à la Dominique : c'est pourquoi on garde la borne sans inventer de valeur.

## 3. Indicateurs calculés et plausibilité

| Indicateur | Formule | Unité |
|---|---|---|
| `Kcal_total` | Σ kcal végétales + Σ kcal animales (tous produits) | kcal/personne/jour |
| `Part_kcal_animal_pct` | kcal animales / kcal totales | % |
| `Part_kcal_cereales_pct` | kcal des 9 céréales du fichier céréales / kcal totales | % |
| `Part_prot_animal_pct` | protéines animales / protéines totales | % |
| `Taux_sous_nutrition_pct` | personnes sous-alimentées / population × 100 | % |

In [6]:
cols = ["Kcal_total", "Part_kcal_animal_pct", "Part_kcal_cereales_pct", "Part_prot_animal_pct",
        "Prot_total_g", "Population_millions", "Taux_sous_nutrition_pct"]
df[cols].describe().T[["count", "mean", "min", "50%", "max"]].round(1)

,count,mean,min,50%,max
Kcal_total,174.00,2 847.50,1 879.00,2 830.50,3 770.00
Part_kcal_animal_pct,174.00,18.60,2.80,18.40,44.30
Part_kcal_cereales_pct,174.00,40.30,17.40,38.10,76.50
Part_prot_animal_pct,174.00,42.80,9.80,45.10,73.10
Prot_total_g,174.00,81.30,37.70,81.60,133.10
Population_millions,174.00,40.20,0.10,9.40,1 385.60
Taux_sous_nutrition_pct,96.00,16.40,3.40,12.20,50.40


In [7]:
print("Plausibilité : les 3 plus faibles et les 3 plus fortes disponibilités caloriques")
print(df.nsmallest(3, "Kcal_total")[["Zone", "Kcal_total"]].to_string(index=False))
print(df.nlargest(3, "Kcal_total")[["Zone", "Kcal_total"]].to_string(index=False))
print(f"\nPopulation totale couverte : {df['Population_millions'].sum():,.0f} millions (sans double compte de la Chine)")

Plausibilité : les 3 plus faibles et les 3 plus fortes disponibilités caloriques
                     Zone  Kcal_total
République centrafricaine      1879.0
                   Zambie      1924.0
               Madagascar      2056.0
    Zone  Kcal_total
Autriche      3770.0
Belgique      3737.0
 Turquie      3708.0

Population totale couverte : 6,997 millions (sans double compte de la Chine)


**Lecture.** De 1 879 kcal (République centrafricaine) à 3 770 kcal (Autriche) : l'ordre de grandeur est
cohérent avec les besoins (≈ 2 000 à 2 500 kcal par adulte) et avec les ordres de grandeur publiés par la FAO.
Les 174 zones couvrent 7,0 milliards de personnes, contre 8,4 avec l'ancien double compte.

## 4. Export

In [8]:
chemin = fp.exporter(df, RACINE / "data" / "global_fao_dataset.csv")
print(f"Exporté : {chemin.relative_to(RACINE)}")
df.head()

Exporté : data/global_fao_dataset.csv


,Code zone,Zone,Population_milliers,Population_millions,Kcal_veg,Kcal_anim,Kcal_cereales,Prot_veg_g,Prot_anim_g,Kcal_total,Prot_total_g,Part_kcal_animal_pct,Part_kcal_cereales_pct,Part_prot_animal_pct,Symbole_sous_alim,Sous_alim_borne,Sous_alimentation_millions,Taux_sous_nutrition_pct,Taux_max_pct,Statut_sous_nutrition,ISO3,Region
0,2,Afghanistan,30552,30.55,1 871.00,216.00,1 560.00,46.05,12.21,2 087.00,58.26,10.35,74.75,20.96,F,False,7.90,25.86,NaN,mesurée,AFG,Asie centrale et du Sud
1,202,Afrique du Sud,52776,52.78,2 533.00,487.00,1 538.00,48.94,36.39,3 020.00,85.33,16.13,50.93,42.65,F,False,2.60,4.93,NaN,mesurée,ZAF,Afrique subsaharienne
2,3,Albanie,3173,3.17,2 203.00,985.00,1 144.00,51.96,59.41,3 188.00,111.37,30.90,35.88,53.34,F,False,0.20,6.30,NaN,mesurée,ALB,Europe et Amérique du Nord
3,4,Algérie,39208,39.21,2 915.00,378.00,1 678.00,66.94,24.98,3 293.00,91.92,11.48,50.96,27.18,F,False,1.70,4.34,NaN,mesurée,DZA,Afrique du Nord et Asie de l'Ouest
4,79,Allemagne,82727,82.73,2 461.00,1 042.00,872.00,39.90,61.49,3 503.00,101.39,29.75,24.89,60.65,NR,False,NaN,NaN,NaN,non rapportée (NR),DEU,Europe et Amérique du Nord
